<a href="https://colab.research.google.com/github/jzamcaz/IPC_JZ_2025/blob/main/mvp_triaje_inteligente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install streamlit pyngrok openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 66.6 MB/s eta 0:00:00


In [4]:
!pip install streamlit-geolocation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.7/802.7 kB 27.4 MB/s eta 0:00:00


In [9]:
%%writefile triage_app.py
import streamlit as st
import pandas as pd
import numpy as np
import unicodedata
from streamlit_geolocation import streamlit_geolocation

# --- 1. Configuración de página ---
st.set_page_config(page_title="MVP Triaje Inteligente", page_icon="🏥", layout="wide")
st.title("🏥 Sistema de Triaje Inteligente & Derivación Médica")

# --- Inicializar ubicación en session_state (persiste entre re-ejecuciones) ---
if "user_lat" not in st.session_state:
    st.session_state.user_lat = -0.1807   # Quito Centro por defecto
    st.session_state.user_lon = -78.4678
    st.session_state.ubicacion_real = False

# --- 2. Carga y Limpieza de Datos ---
def _normaliza(texto):
    texto = str(texto).strip().lower()
    return unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii')

@st.cache_data
def cargar_Establecimiento():
    df = pd.read_excel("Base_Casas_Salud_Quito_Emergencias_v1.xlsx")
    df.columns = df.columns.str.strip()

    # --- Mapa de posibles nombres -> nombre estándar que usa el código ---
    posibles = {
        "institucion":        "Institución",
        "nivel de complejidad":"Nivel de complejidad",
        "establecimiento":    "Establecimiento",
        "nombre":             "Establecimiento",   # por si se llama "nombre"
        "latitud":            "Latitud",
        "latitude":           "Latitud",
        "lat":                "Latitud",
        "longitud":           "Longitud",
        "longitude":          "Longitud",
        "lon":                "Longitud",
        "lng":                "Longitud",
        "long":               "Longitud",
    }

    renombrar = {}
    for col in df.columns:
        clave = _normaliza(col)
        if clave in posibles:
            renombrar[col] = posibles[clave]
    df = df.rename(columns=renombrar)

    # --- Validar que existan las columnas críticas ---
    requeridas = {'Institución', 'Nivel de complejidad', 'Establecimiento', 'Latitud', 'Longitud'}
    faltan = requeridas - set(df.columns)
    if faltan:
        st.error(f"❌ Faltan estas columnas en el Excel (revisa los nombres): {faltan}")
        st.write("Columnas encontradas:", list(df.columns))
        st.stop()

    # --- Limpieza ---
    df['Institución'] = df['Institución'].astype(str).str.strip().str.upper()
    df['Nivel de complejidad'] = df['Nivel de complejidad'].astype(str).str.strip().str.upper()

    # Convertir coordenadas a número (por si vienen como texto con comas)
    for c in ['Latitud', 'Longitud']:
        df[c] = (df[c].astype(str)
                       .str.replace(',', '.', regex=False)  # "‑0,18" -> "‑0.18"
                       .str.strip())
        df[c] = pd.to_numeric(df[c], errors='coerce')

    # Eliminar filas sin coordenadas válidas
    df = df.dropna(subset=['Latitud', 'Longitud']).reset_index(drop=True)

    return df
# Cargar el DataFrame de hospitales una sola vez al iniciar la app
df_hospitales = cargar_Establecimiento()

# --- 3. Funciones de Cálculo ---
def calcular_distancia(lat1, lon1, lat2, lon2):
    """Distancia Haversine en km entre dos coordenadas."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

def evaluar_triaje(edad, temp, fc, sintomas, glasgow, fast_alerta, iam_alerta):
    """Clasificación tipo Manchester por prioridad descendente."""
    sintomas_rojos = ["Dolor Torácico opresivo / Sospecha de Infarto", "Dificultad Respiratoria Severa",
                      "Hemorragia / Sangrado activo abundante", "Convulsiones activas",
                      "Traumatismo mayor (Accidente, caída > 2m)"]
    if glasgow <= 8 or fast_alerta or iam_alerta or any(s in sintomas for s in sintomas_rojos):
        return {"color": "#FF0000", "nivel": "Rojo", "desc": "Emergencia Vital (Atención Inmediata)", "comp": "III"}

    sintomas_naranjas = ["Dificultad Respiratoria Moderada", "Fiebre > 39°C o persistente",
                         "Reacción alérgica severa (hinchazón)", "Quemaduras graves o extensas"]
    if temp >= 39.5 or fc > 130 or fc < 45 or any(s in sintomas for s in sintomas_naranjas):
        return {"color": "#FF8C00", "nivel": "Naranja", "desc": "Muy Urgente (Tiempo límite: 10-15 min)", "comp": "III"}

    sintomas_amarillos = ["Dolor Abdominal Agudo", "Cefalea (Dolor de cabeza) intensa o súbita",
                          "Vómitos incontrolables o con sangre", "Pérdida de conocimiento previa / Desmayo",
                          "Crisis psiquiátrica / Agitación severa"]
    if edad < 2 or edad > 65 or any(s in sintomas for s in sintomas_amarillos):
        return {"color": "#FFFF00", "nivel": "Amarillo", "desc": "Urgente (Tiempo límite: 60 min)", "comp": "II"}

    sintomas_azules = ["Chequeo general / Control", "Renovación de recetas", "Certificado médico o trámite administrativo"]
    if any(s in sintomas for s in sintomas_azules) and temp < 37.5 and len(sintomas) <= 1:
        return {"color": "#0000FF", "nivel": "Azul", "desc": "No Urgente (Consulta Externa / 240 min)", "comp": "I"}

    if len(sintomas) > 0 or temp >= 37.5:
        return {"color": "#008000", "nivel": "Verde", "desc": "Estándar (Tiempo límite: 120 min)", "comp": "I"}

    return {"color": "#0000FF", "nivel": "Azul", "desc": "No Urgente (Consulta Externa / 240 min)", "comp": "I"}

# --- 4. Interfaz Principal ---
col_principal, col_resultados = st.columns([1.2, 1])

with col_principal:
    st.subheader("🚨 Escalas Clínicas Visuales (Llenar si aplica)")

    # ----- FAST -----
    with st.expander("🧠 Escala FAST (Identificación de ACV - Derrame Cerebral)", expanded=False):
        st.markdown("Marque si el paciente presenta **alguno** de estos signos físicos:")
        c1, c2, c3 = st.columns(3)
        with c1:
            st.markdown("<h1 style='text-align: center;'>🫠</h1>", unsafe_allow_html=True)
            f = st.checkbox("**F**ace (Cara asimétrica)")
        with c2:
            st.markdown("<h1 style='text-align: center;'>💪</h1>", unsafe_allow_html=True)
            a = st.checkbox("**A**rms (Debilidad brazo)")
        with c3:
            st.markdown("<h1 style='text-align: center;'>👅</h1>", unsafe_allow_html=True)
            s = st.checkbox("**S**peech (Dificultad hablar)")
        fast_alerta = f or a or s
        if fast_alerta:
            st.error("⚠️ SIGNOS POSITIVOS DE ACV. El sistema forzará la prioridad **ROJA**.")

    # ----- IAM -----
    with st.expander("🫀 Escala IAM (Infarto Agudo de Miocardio)", expanded=False):
        st.markdown("Evalúe las características si el paciente presenta dolor torácico:")
        st.markdown("#### 💥 1. Tipo de Dolor")
        tipo_dolor = st.radio(
            "Sensación principal:",
            ["Ninguno / Otro", "Punzante (como pinchazos)",
             "Opresivo (como un peso aplastante o elefante en el pecho)"],
            horizontal=True
        )

        st.markdown("#### ⚡ 2. Irradiación (¿Hacia dónde viaja el dolor?)")
        c_irr1, c_irr2 = st.columns(2)
        with c_irr1:
            irr_mandibula_cuello = st.checkbox("Cuello o Mandíbula")
        with c_irr2:
            irr_brazo_izq = st.checkbox("Hombros o Brazo Izquierdo")

        st.markdown("#### ⏱️ 3. Duración y Síntomas Asociados")
        duracion_dolor = st.slider("Duración aproximada del dolor (minutos)", 0, 120, 0, step=5)

        c_sint1, c_sint2, c_sint3 = st.columns(3)
        with c_sint1:
            iam_resp = st.checkbox("Dificultad para respirar 😮‍💨")
        with c_sint2:
            iam_diaforesis = st.checkbox("Sudoración fría / Diaforesis 😰")
        with c_sint3:
            iam_conciencia = st.checkbox("Pérdida de conciencia 😵")

        iam_alerta = False
        opresivo = tipo_dolor == "Opresivo (como un peso aplastante o elefante en el pecho)"
        if opresivo and (irr_mandibula_cuello or irr_brazo_izq) and (iam_diaforesis or iam_conciencia):
            iam_alerta = True
            st.error("⚠️ ALTA PROBABILIDAD DE INFARTO AGUDO DE MIOCARDIO. El sistema forzará la prioridad **ROJA**.")
        elif opresivo and duracion_dolor > 15:
            st.warning("⚠️ Dolor opresivo prolongado. Evaluar urgencia coronaria.")

    # ----- GLASGOW -----
    with st.expander("📊 Escala de Glasgow (Nivel de Conciencia)", expanded=False):
        st.markdown("Seleccione la mejor respuesta del paciente:")
        st.markdown("#### 👀 1. Apertura Ocular")
        ojo = st.radio("¿Cómo abre los ojos?", [4, 3, 2, 1],
                       format_func=lambda x: {4:"4 - Espontáneamente (Normal)", 3:"3 - Al hablarle",
                                              2:"2 - Al causarle dolor", 1:"1 - No los abre"}[x], horizontal=True)
        st.markdown("#### 🗣️ 2. Respuesta Verbal")
        verbal = st.radio("¿Cómo responde al hablarle?", [5, 4, 3, 2, 1],
                          format_func=lambda x: {5:"5 - Orientado y conversa", 4:"4 - Desorientado/Confuso",
                                                 3:"3 - Palabras inapropiadas", 2:"2 - Sonidos", 1:"1 - Sin respuesta"}[x], horizontal=True)
        st.markdown("#### 🏃 3. Respuesta Motora")
        motor = st.radio("¿Cómo se mueve?", [6, 5, 4, 3, 2, 1],
                         format_func=lambda x: {6:"6 - Obedece órdenes", 5:"5 - Localiza el dolor",
                                                4:"4 - Se retira al dolor", 3:"3 - Flexión anormal",
                                                2:"2 - Extensión anormal", 1:"1 - Sin respuesta"}[x], horizontal=True)

        total_glasgow = ojo + verbal + motor
        st.metric(label="Puntaje Glasgow Total", value=f"{total_glasgow}/15")
        if total_glasgow <= 8:
            st.error("⚠️ Glasgow ≤ 8 (Coma/Trauma severo). El sistema forzará la prioridad **ROJA**.")

    st.markdown("---")

    # ===== UBICACIÓN (FUERA DEL FORM porque usa un componente interactivo) =====
    st.subheader("📍 Ubicación del Paciente")
    st.caption("Haz clic en el ícono para compartir tu ubicación (acepta permisos del navegador).")
    location = streamlit_geolocation()
    if location and location.get('Latitud') is not None:
        st.session_state.user_lat = location['Latitud']
        st.session_state.user_lon = location['Longitud']
        st.session_state.ubicacion_real = True

    if st.session_state.ubicacion_real:
        st.success(f"✅ Ubicación real: {st.session_state.user_lat:.4f}, {st.session_state.user_lon:.4f}")
    else:
        st.info("ℹ️ Usando ubicación predeterminada (Quito Centro).")

    st.markdown("---")
    st.subheader("📋 Datos Básicos del Paciente")

    # ===== FORMULARIO (solo inputs + submit) =====
    with st.form("triage_form"):
        c_edad, c_temp, c_fc = st.columns(3)
        with c_edad:
            edad = st.number_input("Edad (Años)", 0, 120, 30)
        with c_temp:
            temp = st.number_input("Temperatura (°C)", 34.0, 43.0, 36.5)
        with c_fc:
            fc = st.number_input("Frecuencia Cardíaca (LPM)", 30, 220, 75)

        lista_sintomas_completa = [
            "Dolor Torácico opresivo / Sospecha de Infarto", "Dificultad Respiratoria Severa",
            "Hemorragia / Sangrado activo abundante", "Convulsiones activas", "Traumatismo mayor (Accidente, caída > 2m)",
            "Dificultad Respiratoria Moderada", "Fiebre > 39°C o persistente", "Reacción alérgica severa (hinchazón)",
            "Quemaduras graves o extensas", "Dolor Abdominal Agudo", "Cefalea (Dolor de cabeza) intensa o súbita",
            "Vómitos incontrolables o con sangre", "Pérdida de conocimiento previa / Desmayo", "Crisis psiquiátrica / Agitación severa",
            "Tos leve / Congestión nasal", "Dolor de garganta", "Dolor muscular o articular leve",
            "Erupción cutánea sin fiebre", "Molestia urinaria leve", "Traumatismo o golpe leve",
            "Chequeo general / Control", "Renovación de recetas", "Certificado médico o trámite administrativo", "Otro malestar general leve"
        ]
        sintomas = st.multiselect("Seleccione síntomas generales adicionales presentes:", lista_sintomas_completa)

        st.markdown("---")
        st.subheader("🛡️ Cobertura / Seguro")
        seguro = st.selectbox("Cobertura / Seguro", ["IESS", "MSP", "Privado", "ISSFA", "ISSPOL"])

        enviar = st.form_submit_button("EVALUAR Y BUSCAR HOSPITALES", use_container_width=True)

# --- 5. Lógica de Resultados ---
with col_resultados:
    st.subheader("🩺 Resultados")
    if enviar:
        # Leemos la ubicación desde session_state
        user_lat = st.session_state.user_lat
        user_lon = st.session_state.user_lon

        seguro_buscado = seguro.strip().upper()
        res = evaluar_triaje(edad, temp, fc, sintomas, total_glasgow, fast_alerta, iam_alerta)

        # Alerta visual
        texto_negro = res['color'] in ['#FFFF00', '#008000']
        color_texto = 'black' if texto_negro else 'white'
        st.markdown(
            f"""
            <div style="background-color:{res['color']}; padding:20px; border-radius:10px; text-align:center; box-shadow: 0 4px 6px rgba(0,0,0,0.3);">
                <h2 style="color:{color_texto}; margin:0;">Nivel: {res['nivel']}</h2>
                <h4 style="color:{color_texto}; margin:10px 0 0 0;">{res['desc']}</h4>
            </div>
            """, unsafe_allow_html=True
        )

        st.markdown("---")
        st.subheader("🗺️ Hospitales Sugeridos")

        hospitales_filtrados = df_hospitales[df_hospitales['Institución'] == seguro_buscado].copy()

        if res['comp'] == "III":
            hospitales_filtrados = hospitales_filtrados[hospitales_filtrados['Nivel de complejidad'] == "III"]
        elif res['comp'] == "II":
            hospitales_filtrados = hospitales_filtrados[hospitales_filtrados['Nivel de complejidad'].isin(["II", "III"])]

        if not hospitales_filtrados.empty:
            hospitales_filtrados['Distancia (Km)'] = hospitales_filtrados.apply(
                lambda row: calcular_distancia(user_lat, user_lon, row['Latitud'], row['Longitud']), axis=1
            )
            hospitales_filtrados = hospitales_filtrados.sort_values(by='Distancia (Km)').head(3)

            for idx, row in hospitales_filtrados.iterrows():
                with st.container():
                    st.markdown(f"**🏥 {row['Establecimiento']}** (Nivel {row['Nivel de complejidad']})")
                    st.write(f"📍 A **{row['Distancia (Km)']:.2f} km** de tu ubicación.")
                    url_maps = (f"https://www.google.com/maps/dir/?api=1&origin={user_lat},{user_lon}"
                                f"&destination={row['Latitud']},{row['Longitud']}&travelmode=driving")
                    st.markdown(f"[🗺️ Iniciar Navegación en Google Maps]({url_maps})")
                    st.markdown("---")

            st.map(hospitales_filtrados[['Latitud', 'Longitud']].rename(columns={'Latitud': 'lat', 'Longitud': 'lon'}))
        else:
            st.warning(f"No se encontraron unidades de la red {seguro} que cumplan con la complejidad {res['comp']}.")
    else:
        st.info("Complete el formulario y presione el botón para evaluar.")


Overwriting triage_app.py


In [7]:
import time
import subprocess
from pyngrok import ngrok

# 1. Aseguramos que no haya túneles previos activos
ngrok.kill()

# Set the authentication token before connecting
ngrok.set_auth_token("3G13L0y0MkI7DKDdIVbcWGcNH2a_2aatrzGngTx9i7FaBWEJ9")

# 2. Iniciamos Streamlit en segundo plano
# El proceso se ejecutará mientras el notebook sigue respondiendo
process = subprocess.Popen(['streamlit', 'run', 'triage_app.py', '--server.port', '8501'])

# 3. Esperamos 5 segundos a que Streamlit levante totalmente
print("Iniciando Streamlit...")
time.sleep(5)

# 4. Ahora conectamos ngrok
public_url = ngrok.connect(8501)
print(f"✅ ¡Éxito! Tu app está corriendo en: {public_url}")

Iniciando Streamlit...
✅ ¡Éxito! Tu app está corriendo en: NgrokTunnel: "https://wager-timid-annually.ngrok-free.dev" -> "http://localhost:8501"


In [67]:
from pyngrok import ngrok
ngrok.set_auth_token("3G13L0y0MkI7DKDdIVbcWGcNH2a_2aatrzGngTx9i7FaBWEJ9")

In [ ]:
import pandas as pd
df_test = pd.read_excel("Base_Casas_Salud_Quito_Emergencias_v1.xlsx")
print("Las cabeceras reales del Excel son:")
print(df_test.columns.tolist())

Las cabeceras reales del Excel son:
['ID', 'Establecimiento', 'Institución', 'Red', 'Tipología', 'Nivel de atención', 'Nivel de complejidad', 'Emergencia 24 h', 'Observaciones']


In [86]:
# =========================
# 1. LIMPIAR PROCESOS
# =========================
!pkill -f streamlit
!pkill -f ngrok

import time
import subprocess
from pyngrok import ngrok
import requests

PORT = 8502

# =========================
# 2. LEVANTAR STREAMLIT (VISIBLE LOGS)
# =========================
process = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", str(PORT),
     "--server.address", "0.0.0.0"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# =========================
# 3. ESPERAR ARRANQUE REAL
# =========================
print("⏳ Iniciando Streamlit...")

streamlit_ready = False

for i in range(20):  # ~20 segundos max
    try:
        r = requests.get(f"http://localhost:{PORT}")
        if r.status_code == 200:
            streamlit_ready = True
            break
    except:
        pass
    time.sleep(1)

if not streamlit_ready:
    print("❌ Streamlit no arrancó correctamente")
    print("📄 Logs:")
    print(process.stdout.read())
    print(process.stderr.read())
else:
    print("✅ Streamlit activo")

# =========================
# 4. NGROK SOLO SI STREAMLIT FUNCIONA
# =========================
if streamlit_ready:
    public_url = ngrok.connect(PORT)
    print("🚀 URL pública:")
    print(public_url)

⏳ Iniciando Streamlit...
❌ Streamlit no arrancó correctamente
📄 Logs:

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py

